In [1]:
from IPython import get_ipython
from IPython.display import display
# %%
import cv2
import pytesseract
import pandas as pd
import re
import difflib
import time
from typing import List, Dict

# %%
class HarmfulIngredientDetector:
    def __init__(self, ingredients_data: pd.DataFrame = None, csv_path: str = None, excel_path: str = None):
        if ingredients_data is not None:
            self.ingredients_db = ingredients_data
        elif csv_path is not None:
            self.ingredients_db = pd.read_csv(csv_path)
        elif excel_path is not None:
            self.ingredients_db = pd.read_excel(excel_path)
        else:
            raise ValueError("Either ingredients_data, csv_path, or excel_path must be provided")

        self.ingredients_db['ingredient'] = self.ingredients_db['ingredient'].str.lower()
        self.ingredient_classes = {
            'harmful': set(self.ingredients_db[self.ingredients_db['class'] == 'harmful']['ingredient'].str.lower()),
            'controversial': set(self.ingredients_db[self.ingredients_db['class'] == 'controversial']['ingredient'].str.lower()),
            'not harmful': set(self.ingredients_db[self.ingredients_db['class'] == 'not harmful']['ingredient'].str.lower())
        }

    # Define _preprocess_ingredient as an instance method
    def _preprocess_ingredient(self, ingredient: str) -> str:
        cleaned = re.sub(r'\([^)]*\)', '', ingredient)
        cleaned = re.sub(r'[^a-zA-Z0-9\s-]', '', cleaned)
        return cleaned.strip().lower()

    # Define _find_direct_matches as an instance method
    def _find_direct_matches(self, ingredients: List[str]) -> List[Dict]:
        matches = []
        for ing in ingredients:
            cleaned = self._preprocess_ingredient(ing)
            classification = next((key for key, value in self.ingredient_classes.items() if cleaned in value), None)
            if classification:
                matches.append({
                    'original': ing,
                    'matched_as': cleaned,
                    'classification': classification,
                    'match_type': 'direct'
                })
        return matches
    
    # Define analyze_ingredients as an instance method
    def analyze_ingredients(self, ingredient_list: List[str]) -> Dict:
        results = { 'harmful': [], 'controversial': [], 'not harmful': [], 'unknown': [] }
        direct_matches = self._find_direct_matches(ingredient_list)
        categorized_ingredients = {match['original'] for match in direct_matches}

        for match in direct_matches:
            results[match['classification']].append(match)

        for ing in ingredient_list:
            if ing not in categorized_ingredients:
                results['unknown'].append({'original': ing})

        return results

    # Define scan_and_analyze as an instance method
    def scan_and_analyze(self):
        print("Starting camera. Press 's' to capture an image and analyze ingredients, or 'q' to quit.")
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            print("Error: Could not open camera.")
            return

        while True:
            ret, frame = cap.read()
            if not ret:
                print("Error: Failed to capture image.")
                break

            cv2.imshow("Ingredient Scanner", frame)
            key = cv2.waitKey(1) & 0xFF

            if key == ord('s'):
                filename = f"captured_label_{int(time.time())}.jpg"
                cv2.imwrite(filename, frame)
                print("Image captured! Processing...")
                cap.release()
                cv2.destroyAllWindows()
                self._process_image(filename)
                break
            elif key == ord('q'):
                print("Exiting...")
                cap.release()
                cv2.destroyAllWindows()
                break
                
    # Define _process_image as an instance method
    def _process_image(self, image_path: str):
        img = cv2.imread(image_path)
        if img is None:
            print(f"Error: Could not load image {image_path}")
            return

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        extracted_text = pytesseract.image_to_string(gray)
        ingredient_list = re.findall(r'[A-Za-z]+(?:[-\s][A-Za-z]+)*', extracted_text)

        print("Extracted Ingredients:", ingredient_list)
        results = self.analyze_ingredients(ingredient_list)
        print("Analysis Results:", results)
        
# %%
if __name__ == "__main__":
    csv_path = "ing_data.csv"
    detector = HarmfulIngredientDetector(csv_path=csv_path)
    detector.scan_and_analyze()


Starting camera. Press 's' to capture an image and analyze ingredients, or 'q' to quit.
Image captured! Processing...
Extracted Ingredients: ['RIBOFLAVIN', 'FOLIC ACID', 'SEMI-SWEET CHOC', 'SUGAR', 'CHOCOLATE LIQUOR', 'COCOA BUTTER', 'SO', 'EMULSIFIER', 'VANILLA', 'COCOA', 'PROCESSED WITH ALKALI', 'CANOLA OR SOYBEAN OIL', 'BITTERSWEET CHOCOLATE CHIPS', 'CHOCOLATE LIQUOR', 'SUGAR', 'COCOA BUTTER', 'MILK FAT', 'SOY LECITHIN EMULSIFIER', 'VANILLA', 'MILK CHOCOLATE\nCHIPS', 'SUGAR', 'TATU MILK POWDER', 're he']
Analysis Results: {'harmful': [], 'controversial': [{'original': 'MILK FAT', 'matched_as': 'milk fat', 'classification': 'controversial', 'match_type': 'direct'}], 'not harmful': [{'original': 'RIBOFLAVIN', 'matched_as': 'riboflavin', 'classification': 'not harmful', 'match_type': 'direct'}, {'original': 'FOLIC ACID', 'matched_as': 'folic acid', 'classification': 'not harmful', 'match_type': 'direct'}, {'original': 'CHOCOLATE LIQUOR', 'matched_as': 'chocolate liquor', 'classificatio